# Metrici de evaluare

Material ONIA — Resurse educaționale. Exercițiile pot fi modificate și executate în JupyterLab.

## De ce avem nevoie de metrici?

Imaginăm un program care estimează prețul unor locuințe. Pentru cinci apartamente, modelul oferă predicții. Două modele pot greși în moduri diferite: unul poate avea mai multe greșeli mici, altul poate greși grav pentru un singur apartament. O singură cifră, numită metrică , rezumă un anumit aspect al calității predicțiilor. Nicio metrică nu poate descrie perfect toate consecințele unei erori. Înainte să calculezi o metrică, verifică tipul problemei: regresie dacă rezultatul este un număr; clasificare dacă rezultatul este o etichetă. Pentru clasificare binară pot fi relevante probabilitățile estimate, nu doar clasele prezise.

In [ ]:
y_real = [4, 6, 8, 10, 12]
y_pred = [4.4, 5.3, 8.8, 10.2, 10.0]
for real, pred in zip(y_real, y_pred):
    print(f"real={real:4.1f}, pred={pred:4.1f}, eroare={real-pred:+.1f}")

## Separarea datelor: train, validare și test

Un set de antrenare este folosit pentru a determina parametrii modelului. Setul de validare servește la alegerea hiperparametrilor și la compararea modelelor, iar testul final este păstrat neatins până la evaluarea finală. În competiții, scorul public poate deveni indirect un set de validare dacă este consultat de foarte multe ori. Astfel, putem adapta modelul la particularități ale acelui subset fără să ne dăm seama. Regulă practică: pentru orice preprocesare care învață statistici (medie, deviație standard, vocabular, valori de imputare), utilizează fit doar pe train și transform pe validation/test. În scikit-learn, Pipeline ajută la evitarea acestor erori.

In [ ]:
from sklearn.model_selection import train_test_split
X = [[i] for i in range(40)]
y = [2 * i + (i % 4) for i in range(40)]
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42
)
print(len(X_train), len(X_val), len(X_test))

## Eroare, eroare absolută și eroare pătratică

Pentru o observație cu valoarea reală y și predicția ŷ , eroarea semnată este e = y − ŷ . Semnul ne arată dacă am supraestimat sau subestimat. Dacă am face media erorilor semnate, cele pozitive și negative s-ar putea anula! De aceea folosim valoarea absolută |e| sau pătratul e² . Exemplu: erorile +5 și −5 au media 0, dar modelul nu este perfect. MAE va fi 5, iar MSE va fi 25.

In [ ]:
erori = [5, -5]
print(sum(erori) / len(erori))
print(sum(abs(e) for e in erori) / len(erori))
print(sum(e ** 2 for e in erori) / len(erori))

## MAE — Mean Absolute Error

MAE = (1/n) · Σ |yᵢ − ŷᵢ| MAE este media distanțelor absolute dintre rezultate și predicții. Are aceeași unitate de măsură ca ținta: dacă prezicem lei, MAE este în lei; dacă prezicem temperaturi în °C, MAE este în °C. Este mai puțin sensibilă la valori extreme decât MSE, dar nu ne spune dacă modelul supraestimează sau subestimează în medie. Exemplu: pentru erorile 1, −2, 3, MAE = (1+2+3)/3 = 2.

In [ ]:
def mae(reale, prezise):
    n = len(reale)
    if n == 0:
        raise ValueError("Nu există observații")
    total = 0
    for y, p in zip(reale, prezise):
        total += abs(y - p)
    return total / n

print(mae([2, 6, 7], [3, 4, 10]))

## MSE și RMSE — penalizarea greșelilor mari

MSE = (1/n) · Σ(yᵢ − ŷᵢ)² RMSE = √MSE MSE mărește influența greșelilor mari. O eroare de 4 contribuie cu 16, în timp ce o eroare de 1 contribuie cu 1. RMSE revine la unitatea inițială a țintei, fiind mai ușor de interpretat. Nici MSE, nici RMSE nu pot fi negative. În problemele de regresie din concurs, dacă evaluatorul folosește RMSE, nu alege modelul după MAE doar fiindcă are această metrică mai mică.

In [ ]:
def mse(reale, prezise):
    total = 0
    for y, p in zip(reale, prezise):
        total += (y - p) ** 2
    return total / len(reale)

valoare = mse([1, 2, 3], [1, 3, 6])
print(f"MSE = {valoare:.3f}")
print(f"RMSE = {valoare ** 0.5:.3f}")

## R² — comparația cu o predicție constantă

R² = 1 − [Σ(yᵢ − ŷᵢ)²] / [Σ(yᵢ − ȳ)²] În numitor apare eroarea unui model foarte simplu care prezice mereu media valorilor reale din setul evaluat. Dacă R² = 1, predicțiile sunt perfecte; dacă R² = 0, modelul este la fel de bun, după suma pătratelor, ca această referință constantă. R² poate fi negativ atunci când modelul este mai slab decât referința. Atenție: pentru o țintă constantă, expresia matematică standard nu este definită. Instrumentele software pot utiliza convenții explicite pentru acel caz.

In [ ]:
from sklearn.metrics import r2_score
print(r2_score([2, 4, 6], [2, 4, 6]))
print(r2_score([2, 4, 6], [4, 4, 4]))
print(r2_score([2, 4, 6], [10, 10, 10]))

## Accuracy în clasificare

Accuracy = numărul predicțiilor corecte / numărul total Pentru zece imagini clasificate, dacă opt etichete au fost prezise corect, accuracy = 0,8. Totuși, dacă 95% dintre imagini sunt dintr-o singură clasă, un model care prezice mereu acea clasă obține 95% accuracy fără să identifice deloc cealaltă clasă. De aceea trebuie analizată distribuția claselor și, la nevoie, alte metrici precum recall sau balanced accuracy.

In [ ]:
y_true = [0, 0, 0, 0, 0, 1]
y_pred = [0, 0, 0, 0, 0, 0]
corecte = sum(a == b for a, b in zip(y_true, y_pred))
print(f"Accuracy = {corecte / len(y_true):.3f}")

## Precision, recall și F1

Definim clasa de interes drept „pozitivă”. TP : pozitive corecte; FP : false alarme; FN : pozitive ratate; TN : negative corecte. Precision = TP/(TP+FP) Recall = TP/(TP+FN) F1 = 2PR/(P+R) Precision răspunde la „dintre alertele generate, câte sunt corecte?”, iar recall la „din toate cazurile pozitive reale, câte am descoperit?”. F1 este media armonică, folosită când dorim un compromis. La numitor zero, trebuie stabilită o convenție software. Exemplu: într-o problemă de detectare a mesajelor spam, multe alarme false pot fi costisitoare; într-o problemă de detectare a defecțiunilor grave, ratarea cazurilor poate fi mai periculoasă.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
y_true = [1, 1, 1, 1, 0, 0]
y_pred = [1, 1, 0, 0, 1, 0]
print(precision_score(y_true, y_pred))
print(recall_score(y_true, y_pred))
print(f1_score(y_true, y_pred))

## Probabilități: log-loss, ROC-AUC și PR-AUC

Clasificatoarele pot produce probabilități, de exemplu 0,91 pentru clasa pozitivă. Două modele pot prezice aceleași clase, dar cu încredere diferită. Log-loss penalizează probabilitățile atribuite insuficient clasei adevărate, în special când modelul este foarte sigur și greșește. ROC-AUC măsoară calitatea ordonării exemplelor pozitive față de negative de-a lungul pragurilor; PR-AUC / Average Precision evidențiază relația precision–recall și poate fi deosebit de informativă când clasa pozitivă este rară. Nu confunda ROC-AUC cu accuracy și nici PR-AUC cu F1 calculat la un singur prag.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
y_true = [0, 1, 0, 1, 0, 1]
p = [0.12, 0.88, 0.31, 0.70, 0.45, 0.66]
print('ROC AUC:', roc_auc_score(y_true, p))
print('AP:', average_precision_score(y_true, p))
print('Log-loss:', log_loss(y_true, p))

## Media micro, macro și weighted

În clasificarea cu trei sau mai multe clase putem calcula F1 pentru fiecare clasă. Macro tratează clasele egal, indiferent câte exemple au; weighted face media ponderată cu numărul de observații per clasă; micro agregă numărul total de TP/FP/FN înainte de calcul. Dacă avem multe exemple „pisică” și puține „câine”, macro-F1 poate dezvălui performanța slabă pe clasa rară care ar fi mascată de accuracy. Pentru problemele multilabel, interpretarea este diferită și trebuie verificată explicit.

In [ ]:
from sklearn.metrics import classification_report
y_true = [0, 0, 0, 0, 1, 1, 2]
y_pred = [0, 0, 0, 1, 1, 0, 0]
print(classification_report(y_true, y_pred, zero_division=0))

## Metricile din scikit-learn și scorer-ul competiției

Metoda model.score(X, y) nu reprezintă aceeași metrică pentru toate modelele : deseori este accuracy pentru clasificatori și R² pentru regresori. Când competiția cere MAE, macro-F1 sau log-loss, calculează explicit metrica solicitată. La cross-validation, scoring="neg_mean_absolute_error" returnează valori negative deoarece API-ul de scor maximizează valoarea. O valoare -2 este mai bună decât -5: MAE corespunzător este 2, respectiv 5.

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
model = DummyRegressor(strategy='mean')
X = [[1], [2], [3], [4]]
y = [2., 3., 4., 9.]
model.fit(X, y)
pred = model.predict(X)
print('R²:', model.score(X, y))
print('MAE:', mean_absolute_error(y, pred))